# Multiple customer classes in a model

In this example, the model has three customer classes. The customer class determines the arrival distribution and service distribution.

The JSON for this built-in example can be loaded using `json2ciw.datasets.load_three_classes_model`.

## Imports

In [1]:
import json

import ciw
from rich import print

from json2ciw.datasets import load_stroke_pathway_model
from json2ciw.engine import CiwConverter, multiple_replications
from json2ciw.results import summarise_results, summarise_results_by_class, tidy_to_wide_format, tidy_to_wide_format_by_class
from json2ciw.schema import ProcessModel

## Load JSON

In [2]:
json_network = load_stroke_pathway_model()
print(json.dumps(json_network, indent=2))

{
  "name": "Stroke pathway with class-specific reneging",
  "description": "Simple multi-class stroke pathway with shared routing and class-specific arrival, service, and 
reneging distributions.",
  "customer_classes": [
    {
      "name": "tia",
      "label": "TIA"
    },
    {
      "name": "mild_moderate_stroke",
      "label": "Mild/moderate stroke"
    },
    {
      "name": "severe_stroke",
      "label": "Severe stroke"
    }
  ],
  "activities": [
    {
      "name": "Acute Stroke Unit",
      "type": "activity",
      "resource": {
        "name": "Stroke beds",
        "capacity": 12
      },
      "arrival_distribution": {
        "by_class": {
          "tia": {
            "type": "exponential",
            "parameters": {
              "rate": 2.0
            }
          },
          "mild_moderate_stroke": {
            "type": "exponential",
            "parameters": {
              "rate": 1.2
            }
          },
          "severe_stroke": {
            "type": "exponential",
            "parameters": {
              "rate": 0.6
            }
          }
        }
      },
      "service_distribution": {
        "by_class": {
          "tia": {
            "type": "exponential",
            "parameters": {
              "mean": 2.0
            }
          },
          "mild_moderate_stroke": {
            "type": "exponential",
            "parameters": {
              "mean": 4.0
            }
          },
          "severe_stroke": {
            "type": "exponential",
            "parameters": {
              "mean": 6.0
            }
          }
        }
      },
      "renege_distribution": {
        "by_class": {
          "tia": {
            "type": "uniform",
            "parameters": {
              "min": 2.0,
              "max": 6.0
            }
          },
          "mild_moderate_stroke": {
            "type": "uniform",
            "parameters": {
              "min": 4.0,
              "max": 10.0
            }
          },
          "severe_stroke": {
            "type": "uniform",
            "parameters": {
              "min": 6.0,
              "max": 12.0
            }
          }
        }
      }
    },
    {
      "name": "Rehab Unit",
      "type": "activity",
      "resource": {
        "name": "Rehab beds",
        "capacity": 8
      },
      "service_distribution": {
        "by_class": {
          "mild_moderate_stroke": {
            "type": "exponential",
            "parameters": {
              "mean": 8.0
            }
          },
          "severe_stroke": {
            "type": "exponential",
            "parameters": {
              "mean": 14.0
            }
          }
        }
      },
      "renege_distribution": {
        "by_class": {
          "mild_moderate_stroke": {
            "type": "uniform",
            "parameters": {
              "min": 8.0,
              "max": 16.0
            }
          },
          "severe_stroke": {
            "type": "uniform",
            "parameters": {
              "min": 10.0,
              "max": 20.0
            }
          }
        }
      }
    }
  ],
  "transitions": [
    {
      "from": "Acute Stroke Unit",
      "to": "Rehab Unit",
      "probability": 1.0
    },
    {
      "from": "Rehab Unit",
      "to": "Exit",
      "probability": 1.0
    }
  ]
}

## Validate with `ProcessModel`

In [3]:
model_instance = ProcessModel(**json_network)

In [4]:
print(model_instance)

ProcessModel(
    name='Stroke pathway with class-specific reneging',
    description='Simple multi-class stroke pathway with shared routing and class-specific arrival, service, and 
reneging distributions.',
    customer_classes=[
        CustomerClass(name='tia', label='TIA'),
        CustomerClass(name='mild_moderate_stroke', label='Mild/moderate stroke'),
        CustomerClass(name='severe_stroke', label='Severe stroke')
    ],
    activities=[
        Activity(
            name='Acute Stroke Unit',
            type='activity',
            resource=Resource(name='Stroke beds', capacity=12),
            service_distribution=ClassDistributionMap(
                by_class={
                    'tia': Distribution(type='exponential', parameters={'mean': 2.0}),
                    'mild_moderate_stroke': Distribution(type='exponential', parameters={'mean': 4.0}),
                    'severe_stroke': Distribution(type='exponential', parameters={'mean': 6.0})
                }
            ),
            arrival_distribution=ClassDistributionMap(
                by_class={
                    'tia': Distribution(type='exponential', parameters={'rate': 2.0}),
                    'mild_moderate_stroke': Distribution(type='exponential', parameters={'rate': 1.2}),
                    'severe_stroke': Distribution(type='exponential', parameters={'rate': 0.6})
                }
            ),
            renege_distribution=ClassDistributionMap(
                by_class={
                    'tia': Distribution(type='uniform', parameters={'min': 2.0, 'max': 6.0}),
                    'mild_moderate_stroke': Distribution(type='uniform', parameters={'min': 4.0, 'max': 10.0}),
                    'severe_stroke': Distribution(type='uniform', parameters={'min': 6.0, 'max': 12.0})
                }
            )
        ),
        Activity(
            name='Rehab Unit',
            type='activity',
            resource=Resource(name='Rehab beds', capacity=8),
            service_distribution=ClassDistributionMap(
                by_class={
                    'mild_moderate_stroke': Distribution(type='exponential', parameters={'mean': 8.0}),
                    'severe_stroke': Distribution(type='exponential', parameters={'mean': 14.0})
                }
            ),
            arrival_distribution=None,
            renege_distribution=ClassDistributionMap(
                by_class={
                    'mild_moderate_stroke': Distribution(type='uniform', parameters={'min': 8.0, 'max': 16.0}),
                    'severe_stroke': Distribution(type='uniform', parameters={'min': 10.0, 'max': 20.0})
                }
            )
        )
    ],
    transitions=[
        Transition(source='Acute Stroke Unit', target='Rehab Unit', probability=1.0),
        Transition(source='Rehab Unit', target='Exit', probability=1.0)
    ]
)

In [5]:
model_instance.display_diagram(include_resources=False, show_class_arrivals=True)

```mermaid 
graph TD
    Arrivals_Acute_Stroke_Unit_tia("TIA</br>Time between arrivals<br/>Exponential(λ=2.0)")
    Arrivals_Acute_Stroke_Unit_mild_moderate_stroke("Mild/moderate stroke</br>Time between arrivals<br/>Exponential(λ=1.2)")
    Arrivals_Acute_Stroke_Unit_severe_stroke("Severe stroke</br>Time between arrivals<br/>Exponential(λ=0.6)")
    Acute_Stroke_Unit["Acute Stroke Unit</br>Class-specific service distributions (n=3)"]
    Rehab_Unit["Rehab Unit</br>Class-specific service distributions (n=2)"]
 Renege_Acute_Stroke_Unit{{"Renege</br>Class-specific reneging distributions (n=3)"}}
 Renege_Rehab_Unit{{"Renege</br>Class-specific reneging distributions (n=2)"}}
    Exit(["Exit"])

    Arrivals_Acute_Stroke_Unit_tia --> Acute_Stroke_Unit
    Arrivals_Acute_Stroke_Unit_mild_moderate_stroke --> Acute_Stroke_Unit
    Arrivals_Acute_Stroke_Unit_severe_stroke --> Acute_Stroke_Unit
    Acute_Stroke_Unit -.-> Renege_Acute_Stroke_Unit
    Rehab_Unit -.-> Renege_Rehab_Unit
    Acute_Stroke_Unit --> Rehab_Unit
    Rehab_Unit --> Exit 
```

In [6]:
model_instance.save_diagram("example7.mmd", include_resources=False)

In [7]:
model_instance.get_distributions_df()

,Activity,Phase,Customer Class,Customer Class Label,Distribution Type,Parameters
0,Acute Stroke Unit,Arrival,tia,TIA,Exponential,rate=2.0
1,Acute Stroke Unit,Arrival,mild_moderate_stroke,Mild/moderate stroke,Exponential,rate=1.2
2,Acute Stroke Unit,Arrival,severe_stroke,Severe stroke,Exponential,rate=0.6
3,Acute Stroke Unit,Service,tia,TIA,Exponential,mean=2.0
4,Acute Stroke Unit,Service,mild_moderate_stroke,Mild/moderate stroke,Exponential,mean=4.0
5,Acute Stroke Unit,Service,severe_stroke,Severe stroke,Exponential,mean=6.0
6,Acute Stroke Unit,Renege,tia,TIA,Uniform,"min=2.0, max=6.0"
7,Acute Stroke Unit,Renege,mild_moderate_stroke,Mild/moderate stroke,Uniform,"min=4.0, max=10.0"
8,Acute Stroke Unit,Renege,severe_stroke,Severe stroke,Uniform,"min=6.0, max=12.0"
9,Rehab Unit,Service,mild_moderate_stroke,Mild/moderate stroke,Exponential,mean=8.0


In [8]:
model_instance.get_routing_matrix_df()

,Acute Stroke Unit,Rehab Unit,Exit
Source Activity,,,
Acute Stroke Unit,0.0,1.0,0.0
Rehab Unit,0.0,0.0,1.0


In [9]:
model_instance.get_resources_df()

,Resource,Activity,Count
0,Stroke beds,Acute Stroke Unit,12
1,Rehab beds,Rehab Unit,8


## Convert to `ciw` parameters

In [10]:
adapter = CiwConverter(model_instance)
network_params = adapter.generate_params()
print(network_params)

{
    'number_of_servers': [12, 8],
    'arrival_distributions': {
        'tia': [Exponential(rate=2.0), None],
        'mild_moderate_stroke': [Exponential(rate=1.2), None],
        'severe_stroke': [Exponential(rate=0.6), None]
    },
    'service_distributions': {
        'tia': [Exponential(rate=0.5), Deterministic(value=0.0)],
        'mild_moderate_stroke': [Exponential(rate=0.25), Exponential(rate=0.125)],
        'severe_stroke': [Exponential(rate=0.16666666666666666), Exponential(rate=0.07142857142857142)]
    },
    'routing': {
        'tia': [[0.0, 1.0], [0.0, 0.0]],
        'mild_moderate_stroke': [[0.0, 1.0], [0.0, 0.0]],
        'severe_stroke': [[0.0, 1.0], [0.0, 0.0]]
    },
    'reneging_time_distributions': {
        'tia': [Uniform(lower=2.0, upper=6.0), None],
        'mild_moderate_stroke': [Uniform(lower=4.0, upper=10.0), Uniform(lower=8.0, upper=16.0)],
        'severe_stroke': [Uniform(lower=6.0, upper=12.0), Uniform(lower=10.0, upper=20.0)]
    }
}

## Build and run the `ciw` model

In [11]:
network = ciw.create_network(**network_params)
sim = ciw.Simulation(network)
sim.simulate_until_max_time(50)
print("Quick simulation run worked!")

Quick simulation run worked!

## Run the model for multiple replications

In [12]:
df_reps = multiple_replications(
    network,
    model_instance,
    num_reps=5,
    runtime=2880,
    warmup=1440,
    n_jobs=-1,
)

df_reps.head()

,rep,node_id,activity_name,resource_name,resource_capacity,measure_scope,customer_class,n_service,mean_wait,mean_service,mean_Lq,utilisation,n_renege,renege_rate,mean_wait_renege,mean_wait_all
0,0,1,Acute Stroke Unit,Stroke beds,12,overall,All,4849,1.964934,3.390854,8.168795,96.704511,693,0.125045,3.225253,2.122531
1,0,1,Acute Stroke Unit,Stroke beds,12,customer_class,tia,2315,1.672387,1.996748,4.102402,NaN,650,0.219224,3.132129,1.992398
2,0,1,Acute Stroke Unit,Stroke beds,12,customer_class,mild_moderate_stroke,1654,2.196264,4.024550,2.656396,NaN,42,0.024764,4.585467,2.255430
3,0,1,Acute Stroke Unit,Stroke beds,12,customer_class,severe_stroke,880,2.299737,5.867241,1.409997,NaN,1,0.001135,6.626852,2.304649
4,0,2,Rehab Unit,Rehab beds,8,overall,All,3325,13.333513,3.338769,42.783819,99.723137,1488,0.309163,11.609387,12.800478


## Convert to wide format

In [ ]:
# overall results
wide = tidy_to_wide_format(df_reps)
wide.head()

In [ ]:
# results by class = server strokes
wide_by_class = tidy_to_wide_format_by_class(df_reps, customer_class="severe_stroke")
wide_by_class.head()

In [ ]:
# results by class = tias
wide_by_class = tidy_to_wide_format_by_class(df_reps, customer_class="tia")
wide_by_class.head()

## Summarise results

In [ ]:
df_reps.head(2)

In [ ]:
summary = summarise_results(df_reps)
summary.round(1)

In [ ]:
summary_class = summarise_results_by_class(df_reps)
summary_class.round(1)